# Seabed Morphology

This notebook introduces the main ideas of *seabed morphology*: what it is, the kinds of features that make up the seafloor, and how those features are mapped from bathymetry data. It sets the conceptual stage for the analysis notebooks that follow in this cookbook. The framing here follows the morphology classification of Dove et al. (2020) and the semi-automated mapping approach of Huang et al. (2023).

:::{note} Guiding works
This cookbook builds on a community seabed morphology classification scheme and on the Geoscience Australia tools that implement it.

The classification scheme:

Dove, D., Nanson, R., Bjarnadóttir, L., et al. (2020). *A two-part seabed geomorphology classification scheme (v.2); Part 1: morphology features glossary*. Zenodo. https://doi.org/10.5281/zenodo.4075248

The Geoscience Australia tools (GA-SaMMT):

Huang, Z., Nanson, R., McNeil, M., Wenderlich, M., Gafeira, J., Post, A. and Nichol, S. (2023). Rule-based semi-automated tools for mapping seabed morphology from bathymetry data. *Frontiers in Marine Science*. https://dx.doi.org/10.3389/fmars.2023.1236788

Huang, Z., Nanson, R., Nichol, S., Sixsmith, J. (2022). *Geoscience Australia's Semi-automated Morphological Mapping Tools (GA-SaMMT) for Seabed Characterisation*. Commonwealth of Australia (Geoscience Australia). https://dx.doi.org/10.26186/146832
:::

---

## What is Seabed Morphology and Why Does It Matter?

Seabed morphology refers to the physical shape and structure of the ocean floor. It is the geometry of discrete features like seamounts, ridges, canyons, plains, depressions, and mounds as they exist beneath the water. Just as topographic maps describe the lay of the land above water, bathymetric data (measurements of water depth) allows scientists to identify and map these underwater landforms.

Understanding seabed morphology matters for:

1. **Habitat and ecology.** The shape of the seafloor influences where marine organisms live. Mapping morphological features is foundational to identifying and protecting coastal and marine habitats.
2. **Geomorphological interpretation.** The forms we observe on the seafloor are records of the processes that created them: volcanic activity, sediment transport, erosion, and tectonic movement. Reading morphology helps scientists reconstruct Earth history and understand ongoing geological processes.
3. **Seabed stability and sediment dynamics.** Knowing how the seafloor is shaped helps predict how sediment moves, where it accumulates, and how stable the seabed is. This is critical for infrastructure like pipelines and cables, as well as for understanding carbon cycling.
4. **Ocean management and policy.** As demand grows for evidence-based ocean governance, consistent and reproducible morphology maps are essential tools for regulators, conservation planners, and resource managers.

The challenge is scale and complexity: the seafloor contains an enormous variety of overlapping features at vastly different sizes, which is why developing standardized, semi-automated mapping methods is an active area of research.

Seabed characterisation requires the measurement, description, and classification of physical features on the seabed. A key first step in this process is the identification of morphological forms, as derived from bathymetric data.

## Prerequisites

This is a conceptual introduction, so it contains no code and assumes no specific Python libraries. A little background on gridded elevation data will help you connect these ideas to the analysis notebooks that follow.

| Concepts | Importance | Notes |
| --- | --- | --- |
| Bathymetry and digital elevation models | Helpful | What a gridded depth surface represents |
| Basic marine geoscience vocabulary | Helpful | Key terms are introduced here as needed |
| Raster and array data | Helpful | Useful for the hands-on notebooks that follow |

- **Time to learn**: 20 minutes

---

## From Depth Measurements to Landforms

Modern seafloor mapping starts with *bathymetry*: a grid of water-depth measurements, most often collected by multibeam echosounders mounted on ships. Each grid cell holds a depth value, so a bathymetry grid is the underwater equivalent of a land-surface digital elevation model. Everything in this cookbook is derived from that single kind of input.

A bathymetry grid on its own is just a field of numbers. Turning it into a map of named landforms requires two things: a shared vocabulary for the features we expect to find, and a repeatable procedure for delineating and labelling them. The rest of this notebook introduces both.

## Bathymetric Highs and Lows

The most basic distinction in seabed morphology is between features that rise above the surrounding seafloor and features that sit below it. A *bathymetric high* is elevated relative to its neighbours, like a seamount or a ridge. A *bathymetric low* is depressed relative to its neighbours, like a canyon or a depression. This split is the foundation of the whole classification: a feature is first recognised as a high or a low, and only then assigned a specific type.

{numref}`fig-profiles` shows idealised cross-section profiles for the high and low feature types. Reading these profiles is a good way to build intuition for how shape alone distinguishes one feature from another, for example the pointed profile of a cone versus the rounded profile of a knoll, or the steep asymmetry of a trench versus the flat floor of a trough.

:::{figure} images/fig2_feature_profiles.png
:name: fig-profiles
:alt: Idealised cross-section profiles of bathymetric high and low seabed features
:width: 100%

Idealised cross-section profiles of the bathymetric high (top) and low (bottom) features, modified from Dove et al. (2020). The greyed-out features are not implemented by the GA-SaMMT tools. Reproduced from Huang et al. (2023) under CC BY 4.0.
:::

## A Shared Vocabulary of Seafloor Features

To compare maps made by different people in different places, the community needs an agreed set of feature names with consistent definitions. The scheme used here, from Dove et al. (2020), is deliberately *two-part*. Part 1 is *morphology*: the description of a feature by its shape, size, and configuration, independent of how it formed. Part 2 is *geomorphology*: the interpretation of the process that created the feature. This notebook, and the GA-SaMMT tools, focus on Part 1. A ridge is mapped because it is an elongated elevation, not because we have decided it is volcanic or sedimentary in origin.

The full Dove et al. (2020) scheme defines 31 feature types. The GA-SaMMT tools target a subset of 18 that are well suited to semi-automated mapping: ten bathymetric highs and eight bathymetric lows. These are the features you will meet again in the analysis notebooks.

| Bathymetric highs | Short description |
| --- | --- |
| Ridge | An elongated elevation, longer than it is wide, of varying size and gradient. |
| Seamount | A large, prominent elevation rising more than 1,000 m above the surrounding seafloor. |
| Pinnacle | A spire-shaped pillar, taller than it is wide, isolated or rising from a larger feature. |
| Bank | A relatively shallow elevation, lower than a seamount, often in water less than 200 m deep. |
| Plateau | A flat-topped elevation with one or more relatively steep sides. |
| Cone | A conical elevation with a symmetrical, pointed profile. |
| Knoll | A rounded elevation less than 1,000 m high, wider than it is tall, with a regular profile. |
| Hill | An irregular elevation less than 1,000 m high, generally longer than it is tall. |
| Hummock | A small knoll or mound, smaller than knolls and hills, commonly occurring in groups. |
| Mound | A distinct elevation with a variable, often rounded profile, generally less than 500 m high. |

| Bathymetric lows | Short description |
| --- | --- |
| Hole | A sub-circular depression with steep walls and a generally flat floor. |
| Depression | A general closed-contour low, ranging from small features to large basins. |
| Trench | A long, deep, asymmetric low with relatively steep sides. |
| Trough | A long, wide, flat-bottomed low with symmetrical, subparallel sides. |
| Canyon | An elongated, steep-sided low that deepens downslope and is often sinuous. |
| Valley | An elongated low between bathymetric highs that widens and deepens downslope. |
| Channel | An elongated low with flatter floors and more variable cross-sections than gullies or canyons. |
| Gully | A small, steep-sided, low-sinuosity, relatively high-gradient channel. |

:::{note}
Following Dove et al. (2020), these feature names are treated as proper nouns and capitalised when they refer to the formally defined types. In practice, Valley and Channel are difficult to separate from bathymetry alone, so GA-SaMMT groups them into a single Valley/Channel class.
:::

## Mapping Features: A Three-Step Workflow

Given a bathymetry grid and the vocabulary above, how do we actually produce a feature map? The GA-SaMMT approach breaks the problem into three sequential steps, shown in {numref}`fig-workflow`.

1. **Map.** Delineate the outline of every candidate bathymetric high and low as a polygon. This step works directly from the bathymetry grid and its derivatives, using terrain measures such as the Topographic Position Index to decide which cells belong to a high or a low.
2. **Characterise.** For each polygon, compute a set of geometric attributes that describe its shape (for example length-to-width ratio and circularity), its topography (for example depth range and gradient), and its cross-sectional profile (for example symmetry and concavity).
3. **Classify.** Apply a set of rules to those attributes to assign each polygon one of the feature types from the vocabulary. A long, narrow elevation becomes a Ridge; a tall, broad one becomes a Seamount; and so on.

The three steps are independent. You could delineate polygons by hand and still use the Characterise and Classify steps, or feed polygons from another tool into the same rules. That separation is part of what makes the method flexible and reusable.

:::{figure} images/fig1_gasammt_workflow.png
:name: fig-workflow
:alt: Flow chart of the three-step GA-SaMMT mapping workflow
:width: 70%

The three-step semi-automated mapping workflow: Map, Characterise, and Classify. TPI is the Topographic Position Index, LMI is Local Moran's Index, and CI is the Convergence Index. Reproduced from Huang et al. (2023) under CC BY 4.0.
:::

### The Topographic Position Index

One terrain measure does most of the work in the Map step, so it is worth understanding conceptually. The Topographic Position Index (TPI) compares the depth of a cell to the average depth of its surrounding neighbourhood:

:::{math}
:label: eq-tpi

TPI_{x,y} = E_{x,y} - \overline{WD}_{x,y}
:::

Here {math}`E_{x,y}` is the elevation (depth) at the centre cell and {math}`\overline{WD}_{x,y}` is the mean elevation within a neighbourhood window around it. A positive TPI means the cell sits higher than its surroundings, marking a bathymetric high. A negative TPI means it sits lower, marking a bathymetric low. The size of the neighbourhood window sets the *scale* of the features that are detected: a small window finds small features such as hummocks, while a large window finds broad features such as seamounts. Running the analysis at several window sizes is how a single dataset can be mapped at multiple scales.

## Why Semi-Automated Mapping?

Seabed features have traditionally been mapped by hand, with an expert tracing outlines on a shaded-relief image. Manual mapping has real strengths: the analyst brings deep contextual knowledge and can work around gaps in the data. It also has serious weaknesses. It is subjective, it does not scale to the thousands of small features in a complex survey, and different experts produce different maps from the same data.

Huang et al. (2023) made this concrete. Three experienced mappers were asked to map the same area under the same guidelines, and their results varied widely in the number of features they drew, the area those features covered, and even which feature types they assigned. A rule-based, semi-automated method addresses these problems directly. Once the rules and parameters are set, the same input always yields the same output. The method is objective, repeatable, and efficient enough to characterise thousands of features at once, while still letting analysts inject domain knowledge through the parameters they choose.

## Scale and Resolution Matter

The features you can map depend on the resolution of the bathymetry grid. A fine grid of a few metres per cell can resolve small hummocks and pockmarks, whereas a coarse grid of hundreds of metres per cell can only resolve large features such as seamounts and broad valleys. This introduces a depth bias, because deeper areas are usually surveyed at coarser resolution, so their smaller features go unrecorded. Keeping the grid resolution in mind is essential when interpreting any morphology map.

---

## Summary

Seabed morphology is the geometry of discrete features on the ocean floor, derived from bathymetry data. Every feature is first recognised as a bathymetric high or low, then assigned a type from a shared vocabulary of definitions that describe shape rather than origin. Mapping those features from a bathymetry grid follows a three-step workflow of Map, Characterise, and Classify, with the Topographic Position Index doing much of the work of separating highs from lows at a chosen scale. A semi-automated, rule-based approach makes the resulting maps objective, repeatable, and scalable in a way that manual mapping cannot match.

### What's next?

The notebooks that follow turn these concepts into working code. Upcoming notebooks will read and visualize bathymetry data, compute terrain derivatives such as the Topographic Position Index, and walk through delineating, characterising, and classifying seabed features on a real dataset.

## Resources and References

- Dove, D., Nanson, R., Bjarnadóttir, L., et al. (2020). *A two-part seabed geomorphology classification scheme (v.2); Part 1: morphology features glossary*. Zenodo. https://doi.org/10.5281/zenodo.4075248
- Huang, Z., Nanson, R., McNeil, M., Wenderlich, M., Gafeira, J., Post, A. and Nichol, S. (2023). Rule-based semi-automated tools for mapping seabed morphology from bathymetry data. *Frontiers in Marine Science*, 10:1236788. https://dx.doi.org/10.3389/fmars.2023.1236788
- Huang, Z., Nanson, R., Nichol, S., Sixsmith, J. (2022). *Geoscience Australia's Semi-automated Morphological Mapping Tools (GA-SaMMT) for Seabed Characterisation*. Commonwealth of Australia (Geoscience Australia). https://dx.doi.org/10.26186/146832
- Weiss, A. (2001). *Topographic position and landforms analysis*. Poster, ESRI International User Conference, San Diego, CA.

:::{note}
{numref}`fig-workflow` and {numref}`fig-profiles` are reproduced from Huang et al. (2023), an open-access article distributed under the [Creative Commons Attribution License (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).
:::